# Unit 7: Materials Science - Hubbard Benchmark and Toy QPE

**Companion notebook for *What Quantum Computers Are Actually For***

**Notebook class:** toy demonstration


**Code note:** This notebook is written as teaching code rather than library code. The Quokka calls, circuit construction, and measurement post-processing stay explicit on purpose so you can inspect the mechanism end to end. A production library would factor more of this behind helpers; here we keep the moving parts visible.

This notebook does two distinct jobs:

1. solve the 2-site Hubbard model exactly as a classical benchmark;
2. use a compiled 3-bit phase-estimation circuit to show how QPE reads out an eigenphase once controlled time evolution exists.

It does **not** implement controlled $e^{-iHt}$ for the Hubbard Hamiltonian, and it does **not** claim a true Mott transition on two sites.

**What you'll see:**
1. Exact diagonalisation of the 2-site Hubbard benchmark
2. The interaction-driven crossover in that benchmark as $U/t$ increases
3. A shift-and-scale map from one benchmark energy onto a 3-bit phase register
4. A compiled QPE toy circuit that recovers that phase on Quokka

In [ ]:
import json

import numpy as np

import matplotlib.pyplot as plt



import requests

from requests.packages.urllib3.exceptions import InsecureRequestWarning

requests.packages.urllib3.disable_warnings(InsecureRequestWarning)



QUOKKA = "quokka1"

QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"



def _decode_quokka_counts(payload: dict) -> dict:
    """Normalize Quokka responses to a simple {bitstring: count} mapping."""
    if isinstance(payload, dict) and all(isinstance(v, int) for v in payload.values()):
        return payload

    if not isinstance(payload, dict):
        raise TypeError(f"Unexpected Quokka payload type: {type(payload)!r}")

    if payload.get("error_code", 0) not in (0, None):
        raise RuntimeError(f"Quokka error {payload.get('error_code')}: {payload.get('error')}")

    result = payload.get("result")
    if not isinstance(result, dict) or "c" not in result:
        raise ValueError(f"Unexpected Quokka result format: {payload}")

    counts = {}
    for shot in result["c"]:
        bitstring = ''.join(str(int(bit)) for bit in shot)
        counts[bitstring] = counts.get(bitstring, 0) + 1
    return counts

def run_qasm(program: str, shots: int = 1024) -> dict:
    result = requests.post(
        QUOKKA_URL,
        json={"script": program, "count": shots},
        verify=False,
        timeout=30,
    )
    result.raise_for_status()
    return _decode_quokka_counts(result.json())

print(f"Connected to {QUOKKA_URL}")

## 1. The 2-site Hubbard model

$$H = -t \sum_{\sigma} (c_{1\sigma}^\dagger c_{2\sigma} + \text{h.c.}) + U \sum_i n_{i\uparrow} n_{i\downarrow}$$

2 sites, 2 electrons (half-filling), 4 spin-orbitals → 4 qubits (Jordan-Wigner).

In the 2-electron sector, the Hamiltonian has dimension 6. We can diagonalise it
exactly to get the benchmark.

In [ ]:
def hubbard_2site_energies(t_hop: float, U: float) -> np.ndarray:
    """Exact eigenvalues of the 2-site Hubbard model at half-filling.

    In the singlet sector, the Hamiltonian is 3x3:
    |1> = |up down, 0>  (both electrons on site 1)
    |2> = |0, up down>  (both electrons on site 2)
    |3> = (|up,down> - |down,up>) / sqrt(2)  (one on each, singlet)

    H = [[U, 0, -sqrt(2) t],
         [0, U, -sqrt(2) t],
         [-sqrt(2) t, -sqrt(2) t, 0]]
    """
    H = np.array([
        [U, 0, -np.sqrt(2) * t_hop],
        [0, U, -np.sqrt(2) * t_hop],
        [-np.sqrt(2) * t_hop, -np.sqrt(2) * t_hop, 0],
    ])
    return np.sort(np.linalg.eigvalsh(H))


# Sweep U/t
# On two sites this is a crossover in the benchmark spectrum, not a bulk phase transition.
t_hop = 1.0
U_values = np.linspace(0, 10, 50)
ground_energies = []

for U in U_values:
    energies = hubbard_2site_energies(t_hop, U)
    ground_energies.append(energies[0])

ground_energies = np.array(ground_energies)

plt.figure(figsize=(8, 5))
plt.plot(U_values, ground_energies, '-', color='#009688', linewidth=2)
plt.xlabel('$U/t$')
plt.ylabel('Ground-state energy / $t$')
plt.title('2-site Hubbard benchmark: interaction-driven crossover')
plt.axhline(y=0, color='gray', linestyle=':', alpha=0.3)
plt.annotate('More delocalised\nlow-U regime', xy=(0.8, -2.4), fontsize=11, color='#009688')
plt.annotate('More localised\nhigh-U regime', xy=(6.8, -0.55), fontsize=11, color='#FF5722')
plt.tight_layout()
plt.show()

print('Two sites show a crossover in the benchmark spectrum, not a true Mott phase transition.')
print(f"At U/t=0:  E0 = {hubbard_2site_energies(1, 0)[0]:.4f}t")
print(f"At U/t=4:  E0 = {hubbard_2site_energies(1, 4)[0]:.4f}t")
print(f"At U/t=10: E0 = {hubbard_2site_energies(1, 10)[0]:.4f}t")

## 2. Compiled QPE toy for one benchmark eigenphase

The next cell is deliberately narrower than a full Hubbard simulation.

We take one benchmark ground-state energy at $U/t = 4$, shift and rescale it into the interval $[0, 1)$, round it to the nearest 3-bit phase fraction, and feed that phase into a compiled controlled-phase circuit.

That means the ancilla readout is honest about what it is doing: this is a toy demonstration of binary phase extraction, not a claim that we have implemented controlled $e^{-iHt}$ for the Hubbard Hamiltonian.

In [ ]:
ancilla_bits = 3
levels = 2 ** ancilla_bits
U_target = 4.0
E_exact = float(hubbard_2site_energies(t_hop, U_target)[0])

# Build a compact energy window from the exact benchmark sweep.
energy_min = float(ground_energies.min() - 0.1)
energy_max = float(ground_energies.max() + 0.1)

# Map one benchmark energy to a phase in [0, 1), then snap it to the 3-bit grid
# so the compiled toy circuit has a clean target peak.
phase_continuous = (E_exact - energy_min) / (energy_max - energy_min)
phase_integer = int(np.clip(np.rint(phase_continuous * levels), 0, levels - 1))
encoded_phase = phase_integer / levels
phase_angle = 2 * np.pi * encoded_phase
expected_peak = format(phase_integer, f"0{ancilla_bits}b")


def phase_to_energy(phase_fraction: float) -> float:
    return energy_min + phase_fraction * (energy_max - energy_min)


encoded_energy_proxy = phase_to_energy(encoded_phase)

print(f"Benchmark target at U/t = {U_target}: E0 = {E_exact:.4f}t")
print(f"Energy window for the toy encoder: [{energy_min:.4f}, {energy_max:.4f}]t")
print(f"Continuous phase from that window: {phase_continuous:.4f}")
print(f"Encoded 3-bit phase fraction:      {encoded_phase:.4f} ({expected_peak})")
print(f"Nearest energy on that 3-bit grid: {encoded_energy_proxy:.4f}t")

qpe_circuit = f"""OPENQASM 2.0;
include "qelib1.inc";

qreg q[4];  // q[0-2] = ancilla register, q[3] = eigenstate register
creg c[3];

// Prepare the one-qubit eigenstate |1>
x q[3];

// Put the ancilla register into superposition
h q[0];
h q[1];
h q[2];

// Compiled controlled-U^(2^k) operations for the encoded phase
cu1({phase_angle:.6f}) q[2], q[3];
cu1({2 * phase_angle:.6f}) q[1], q[3];
cu1({4 * phase_angle:.6f}) q[0], q[3];

// Inverse QFT on the ancilla register
h q[0];
cu1({-np.pi / 2:.6f}) q[0], q[1];
h q[1];
cu1({-np.pi / 4:.6f}) q[0], q[2];
cu1({-np.pi / 2:.6f}) q[1], q[2];
h q[2];

// Bit reversal swap
cx q[0], q[2];
cx q[2], q[0];
cx q[0], q[2];

measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
"""

results = run_qasm(qpe_circuit, shots=1024)

print("\nCompiled QPE toy measurement results:")
for outcome in sorted(results.keys(), key=lambda bitstring: results[bitstring], reverse=True):
    phase_fraction = int(outcome, 2) / levels
    energy_proxy = phase_to_energy(phase_fraction)
    print(
        f"  {outcome}: {results[outcome]:>4} counts"
        f" -> phase {phase_fraction:.3f}"
        f" -> energy proxy {energy_proxy:.4f}t"
    )

best_outcome = max(results, key=results.get)
peak_probability = results[best_outcome] / sum(results.values())
best_phase_fraction = int(best_outcome, 2) / levels
best_energy_proxy = phase_to_energy(best_phase_fraction)
grid_error = abs(best_energy_proxy - E_exact)

print(f"\nDominant outcome:            {best_outcome} (expected {expected_peak})")
print(f"Recovered phase fraction:    {best_phase_fraction:.4f}")
print(f"Recovered energy proxy:      {best_energy_proxy:.4f}t")
print(f"Exact benchmark energy:      {E_exact:.4f}t")
print(f"Grid approximation error:    {grid_error:.4f}t")
print(f"Peak probability:            {peak_probability:.3f}")
print("This demonstrates phase readout after compilation, not full Hubbard Hamiltonian simulation.")

## What's next?

- [Deep-Dive 7 - QPE and Trotterisation](https://github.com/johnazariah/quantum-bottleneck/blob/main/manuscript/14-qpe-trotter.md): the machinery behind real Hamiltonian simulation
- Replace the compiled phase oracle with controlled Trotterised time evolution to move from toy QPE to faithful Hubbard simulation
- Increase phase precision by adding more ancilla qubits or tightening the phase window
- [Unit 8 - Climate & Energy](https://github.com/johnazariah/quantum-bottleneck/blob/main/manuscript/15-climate-energy.md): where precomputed active-space Hamiltonians become the next pipeline step